# Free Photo-to-Talking-Video (Hindi audio) — SadTalker

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parmarharmad-art/Social-ai/blob/main/social-content/Free_Hindi_Photo_LipSync_Colab.ipynb)

This notebook attempts open-source SadTalker inference using one portrait image and a pre-generated Hindi audio file. It includes compatibility handling for a known BasicSR/TorchVision import issue. The upstream project documents Python 3.8 as its tested baseline; current hosted notebook runtimes can differ, so this is best-effort rather than guaranteed. Free GPU access is not guaranteed.

**Input:** one portrait photo + Hindi narration audio. **Output:** talking-photo MP4 only if inference succeeds and both video and audio streams validate. This does not translate text into Hindi audio.

**Phone steps:** open this notebook in Colab → Runtime → Change runtime type → select GPU if available → run cells from top to bottom → upload your photo and audio when asked → download the MP4 at the end.

In [ ]:
import os, subprocess, sys
print('Python:', sys.version)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch check:', repr(e))
from google.colab import files
uploaded = files.upload()  # Select one portrait image and one Hindi narration audio file
print('Uploaded:', list(uploaded.keys()))
from pathlib import Path
images = [p for p in Path('/content').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
audios = [p for p in Path('/content').iterdir() if p.is_file() and p.suffix.lower() in {'.wav','.mp3','.m4a','.ogg'}]
print('Image candidates:', [p.name for p in images])
print('Audio candidates:', [p.name for p in audios])
assert images, 'Upload a portrait image (.jpg/.png/.webp).'
assert audios, 'Upload narration audio (.wav/.mp3/.m4a/.ogg).'


## Install SadTalker
Run the next cell once. This downloads the source and model checkpoints; it may take several minutes.

In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil
repo = Path('/content/SadTalker')
def run_checked(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)
if repo.exists() and (repo / 'inference.py').is_file():
    print('SadTalker checkout already exists; reusing it.')
else:
    if repo.exists(): shutil.rmtree(repo)
    run_checked(['git','clone','--depth','1','https://github.com/OpenTalker/SadTalker.git',repo])
print('Installing ffmpeg...')
run_checked(['bash','-lc','apt-get update -qq && apt-get install -y -qq ffmpeg'])
print('Installing Python packages without the upstream legacy scikit-image pin or reinstalling Colab Torch...')
python_minor = sys.version_info.minor
scipy_pin = 'scipy==1.11.4' if sys.version_info >= (3,12) else 'scipy==1.10.1'
print('Python compatibility selection:', sys.version.split()[0], '|', scipy_pin)
packages = ['numpy==1.26.4','face_alignment==1.3.5','imageio==2.19.3','imageio-ffmpeg==0.4.7','librosa==0.9.2','numba','resampy==0.3.1','pydub==0.25.1',scipy_pin,'kornia==0.6.8','tqdm','yacs==0.1.8','pyyaml','joblib==1.3.2','scikit-image==0.21.0','basicsr==1.4.2','facexlib==0.3.0','gradio','gfpgan','av','safetensors']
run_checked([sys.executable,'-m','pip','install','-q'] + packages)
print('Installed package metadata check (diagnostic only):')
subprocess.run([sys.executable,'-m','pip','check'], check=False)
print('Patching known BasicSR/TorchVision import if present...')
import site
patch_candidates = []
for site_dir in site.getsitepackages(): patch_candidates.extend(Path(site_dir).glob('basicsr/data/degradations.py'))
for patch_file in patch_candidates:
    text = patch_file.read_text()
    old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
    if old in text:
        patch_file.write_text(text.replace(old, 'from torchvision.transforms.functional import rgb_to_grayscale'))
        print('Patched:', patch_file)
print('Downloading model checkpoints (large files; wait until command completes)...')
run_checked(['bash','scripts/download_models.sh'], cwd=repo)
checkpoint_dir = repo/'checkpoints'
needed = ['mapping_00109-model.pth.tar','mapping_00229-model.pth.tar','SadTalker_V0.0.2_256.safetensors','SadTalker_V0.0.2_512.safetensors']
missing = [name for name in needed if not (checkpoint_dir/name).is_file() or (checkpoint_dir/name).stat().st_size < 100_000]
if missing: raise RuntimeError('Missing or incomplete model checkpoints: '+', '.join(missing))
print('Setup completed and required model checkpoint files exist.')


In [ ]:
from pathlib import Path
import subprocess, sys, shutil
files_now = [p for p in Path('/content').iterdir() if p.is_file()]
print('Files in /content:', [p.name for p in files_now])
image_path = input('Type the exact photo filename shown above (example: portrait.jpg): ').strip()
audio_path = input('Type the exact Hindi audio filename shown above (example: hindi.wav): ').strip()
image_file = (Path('/content') / image_path).resolve()
audio_file = (Path('/content') / audio_path).resolve()
assert image_file.parent == Path('/content') and image_file.is_file() and image_file.stat().st_size > 0, f'Photo missing/empty or invalid path: {image_path}'
assert audio_file.parent == Path('/content') and audio_file.is_file() and audio_file.stat().st_size > 0, f'Audio missing/empty or invalid path: {audio_path}'
out_dir = '/content/SadTalker/results/kiran_talking_photo'
Path(out_dir).mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'inference.py', '--driven_audio', str(audio_file), '--source_image', str(image_file), '--result_dir', out_dir, '--still', '--preprocess', 'crop']
print('Running SadTalker; this can take several minutes...')
try:
    subprocess.run(cmd, cwd='/content/SadTalker', check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f'SadTalker inference exited with code {e.returncode}. Read the first error above; do not treat this run as successful.') from e
videos = sorted(Path(out_dir).rglob('*.mp4'), key=lambda p: p.stat().st_mtime, reverse=True)
videos = [p for p in videos if p.stat().st_size > 1000]
assert videos, 'No MP4 produced. Check inference traceback, GPU/runtime compatibility, and model downloads.'
output_path = Path('/content/talking_photo_hindi.mp4')
shutil.copy2(videos[0], output_path)
assert output_path.stat().st_size > 1000, 'Output MP4 is empty or incomplete.'
probe = subprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type,width,height','-of','csv=p=0',str(output_path)], check=True, capture_output=True, text=True)
streams = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
assert any(line.startswith('video,') or line == 'video' for line in streams), f'No valid video stream: {streams}'
assert any(line.startswith('audio,') or line == 'audio' for line in streams), f'No valid audio stream: {streams}'
print('VALIDATED OUTPUT (video + audio):', output_path, 'bytes:', output_path.stat().st_size)
from IPython.display import Video, display
display(Video(str(output_path), embed=True))


In [ ]:
from google.colab import files
assert Path('/content/talking_photo_hindi.mp4').is_file() and Path('/content/talking_photo_hindi.mp4').stat().st_size > 1000
files.download('/content/talking_photo_hindi.mp4')

## Troubleshooting
- `CUDA available: False` means this session has no GPU; choose Runtime → Change runtime type → GPU if offered, then rerun from the top. Free GPU access is not guaranteed.
- Setup intentionally installs compatible `scikit-image` and leaves Colab's preinstalled Torch in place instead of blindly installing every old pinned requirement. Other dependency conflicts can still occur.
- If the package installation fails, stop at that cell and copy the first red error. Do not continue to inference with a partially installed environment.
- The model download cell validates four required checkpoint files and their size. If it fails, inspect download errors; do not run inference with incomplete model files.
- If inference fails, send the final traceback. If no non-empty MP4 is found, the run is not successful.
- SadTalker is open-source, but free GPU availability, runtime duration, and hosting are not guaranteed.